
# AI Stock Market Assistant
### Assignment 1.1 – LangChain + LangGraph

**Project Overview**

The AI Stock Market Assistant is an educational financial research application designed to analyze historical stock market data, retrieve relevant financial news, and generate simulated investment recommendations.

The project combines LangChain, LangGraph, Retrieval-Augmented Generation (RAG), FAISS vector search, and Streamlit.

### Key Features

- Historical stock market data retrieval using Yahoo Finance.
- Financial news collection and processing.
- Semantic search using sentence embeddings and FAISS.
- Specialized multi-agent workflow using LangGraph.
- Shared state management and conditional routing.
- Simulated BUY, SELL, and HOLD recommendations.
- Interactive Streamlit dashboard.

**Disclaimer:** This project is developed for academic and educational purposes. Its simulated recommendations are not financial advice.



## 1. Project Objectives

The primary objectives of this project are:

1. Analyze historical stock market information.
2. Retrieve and process relevant financial news.
3. Implement Retrieval-Augmented Generation using FAISS.
4. Use LangChain for prompt templates, tools, and LLM integration.
5. Implement a multi-agent architecture using LangGraph.
6. Demonstrate state management, nodes, edges, and conditional routing.
7. Generate simulated BUY, SELL, or HOLD recommendations with justification.
8. Develop an interactive application using Streamlit.



## 2. Technology Stack

| Technology | Purpose |
|---|---|
| Python | Core programming language |
| LangChain | Tools, prompts, and LLM orchestration |
| LangGraph | Multi-agent workflow and state management |
| Yahoo Finance (yfinance) | Historical stock data and financial news |
| FAISS | Vector database and similarity search |
| Sentence Transformers | Text embeddings |
| Pandas | Data processing |
| NumPy | Numerical operations |
| Streamlit | Interactive web application |


In [ ]:

import json
import pandas as pd
import numpy as np
import yfinance as yf
import streamlit as st

from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from langgraph.graph import StateGraph, START, END

print("All project libraries imported successfully!")



## 3. Stock Market Data Retrieval and Analysis

The application uses the Yahoo Finance API through the `yfinance` Python library to retrieve historical stock market information.

The stock data retrieval tool is implemented using LangChain's `@tool` decorator.

The tool retrieves:

- Latest closing price
- Highest and lowest prices
- Average trading volume
- Historical percentage change
- Number of trading days

The retrieved information is used by the Market Data Agent for further analysis.


In [ ]:

# Import the stock data retrieval tool

from tools import fetch_stock_data

# Select a stock ticker
ticker = "RELIANCE.NS"

# Fetch six months of historical data
market_result = fetch_stock_data.invoke({
    "ticker": ticker,
    "period": "6mo"
})

# Convert JSON response into Python dictionary
market_data = json.loads(market_result)

# Display results
if market_data.get("success"):

    print("STOCK MARKET ANALYSIS")
    print("=" * 40)

    for key, value in market_data.items():
        print(f"{key}: {value}")

else:
    print("Stock data retrieval failed:")
    print(market_data.get("message"))



### 3.1 Interpretation of Stock Market Data

The retrieved historical data provides an overview of the selected company's stock performance.

The percentage change represents the difference between the first and last available closing prices during the selected period.

The highest and lowest prices indicate the observed trading range, while average trading volume provides information about trading activity.

This information is passed to the Market Analysis Agent for further processing.



## 4. Financial News Retrieval

The application retrieves financial news using the Yahoo Finance platform through the `yfinance` library.

News articles are collected and converted into LangChain Document objects.

Each document contains:
- Article title
- News content
- Publisher information
- Source URL
- Associated stock ticker

The retrieved news provides additional context for understanding company developments and potential market risks.



## 5. Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation (RAG) is a technique that retrieves relevant information from an external knowledge source to provide context for a query.

In this project, RAG is implemented using:

1. Financial news retrieval through Yahoo Finance.
2. Document creation using LangChain.
3. Text embeddings using Sentence Transformers.
4. Vector storage using FAISS.
5. Similarity search to retrieve relevant news articles.

The embedding model used is:

`sentence-transformers/all-MiniLM-L6-v2`

FAISS enables efficient similarity-based retrieval of relevant financial news.


In [ ]:

from rag import (
    fetch_financial_news,
    create_news_vector_store,
    retrieve_relevant_news
)

# Select company
ticker = "RELIANCE.NS"

# Retrieve financial news
news_documents = fetch_financial_news(ticker)

print("Financial News Retrieval")
print("=" * 40)

print("Number of articles retrieved:", len(news_documents))

for i, doc in enumerate(news_documents[:3], 1):
    print(f"\nArticle {i}")
    print("Title:", doc.metadata.get("title"))
    print("Publisher:", doc.metadata.get("publisher"))
    print("URL:", doc.metadata.get("url"))


In [ ]:

# Create vector database from retrieved news

if news_documents:

    vector_store = create_news_vector_store(news_documents)

    print("FAISS vector database created successfully!")

else:

    vector_store = None

    print("No news documents available for indexing.")


In [ ]:

# Define a research question

research_query = "What are the latest business developments and risks?"

# Retrieve relevant news articles

if vector_store is not None:

    relevant_news = retrieve_relevant_news(
        vector_store,
        research_query,
        k=3
    )

    print("Relevant Financial News")
    print("=" * 40)

    for i, doc in enumerate(relevant_news, 1):

        print(f"\nArticle {i}:")
        print(doc.page_content)

else:

    relevant_news = []

    print("News retrieval unavailable.")



### 5.1 RAG Implementation – Observations

The RAG component retrieves financial news relevant to a given research query using semantic similarity search.

Sentence Transformers converts the news documents into numerical embeddings, while FAISS stores and searches these embeddings.

The retrieved documents can subsequently be provided as contextual information to an LLM for financial news analysis.

In the current prototype, retrieval is implemented and tested independently of the LLM.



## 6. LangChain Prompt Engineering

LangChain is used to create structured prompt templates for different analytical tasks.

The project contains three prompt templates:

1. Market Analysis Prompt – guides the analysis of historical stock market data.
2. News Analysis Prompt – guides the examination of retrieved financial news.
3. Final Decision Prompt – combines market and news analysis to produce a simulated recommendation.

These templates define the instructions and input variables that will be used when an LLM is connected.


In [ ]:

from prompts import (
    market_analysis_prompt,
    news_analysis_prompt,
    final_decision_prompt
)

print("LANGCHAIN PROMPT TEMPLATES")
print("=" * 45)

print("\n1. Market Analysis Prompt")
print(market_analysis_prompt)

print("\n2. News Analysis Prompt")
print(news_analysis_prompt)

print("\n3. Final Decision Prompt")
print(final_decision_prompt)

print("\nAll three prompt templates loaded successfully!")



### 6.1 Prompt Engineering – Observations

The three prompt templates separate the analytical responsibilities of the proposed AI system.

This modular approach allows each analytical task to have its own instructions and input variables.

The templates are currently defined and tested independently. Actual LLM-based execution will be enabled after API integration.



## 7. LangGraph Multi-Agent Architecture

LangGraph is used to create and manage the multi-agent workflow.

The application contains five specialized nodes:

1. Market Data Agent
2. Market Analysis Agent
3. News Retrieval Agent
4. News Analysis Agent
5. Final Decision Agent

Each node performs a specific task and communicates through a shared state.

The workflow is compiled into an executable graph using LangGraph's StateGraph.


In [ ]:

from graph import graph, StockState

print("LangGraph workflow imported successfully!")
print("\nRegistered workflow nodes:")

for node in graph.get_graph().nodes:
    print("-", node)


In [ ]:

from IPython.display import display, Markdown

# Generate the LangGraph Mermaid diagram
mermaid_diagram = graph.get_graph().draw_mermaid()

display(Markdown(
    "```mermaid\n" + mermaid_diagram + "\n```"
))



### 7.1 State Management

The workflow uses a shared TypedDict state named `StockState`.

It stores information exchanged between agents, including:

- Stock ticker and historical period.
- Retrieved market data.
- Market data availability status.
- Market analysis.
- Retrieved financial news.
- News analysis.
- Final simulated recommendation.
- Workflow execution status.

The shared state allows each agent to access relevant outputs produced by previous agents.



### 7.2 Conditional Routing

The workflow implements conditional routing after the Market Data Agent.

The routing function checks whether stock market data was successfully retrieved.

- If market data is available, execution moves to the Market Analysis Agent.
- If market data is unavailable, execution moves directly to the News Retrieval Agent.

Both execution paths subsequently proceed through News Analysis and Final Decision.

This demonstrates conditional edges and dynamic workflow execution using LangGraph.



## 8. Complete Multi-Agent Workflow Execution

The compiled LangGraph workflow is executed using a stock ticker, historical period, and research question.

The agents process the available market data and financial news before generating a simulated recommendation.


In [ ]:

# Execute the complete LangGraph workflow

initial_state = {
    "ticker": "RELIANCE.NS",
    "period": "6mo",
    "query": "What are the latest business developments and risks?"
}

workflow_result = graph.invoke(initial_state)

print("WORKFLOW EXECUTION RESULTS")
print("=" * 45)

print("\nWorkflow Status:")
print(workflow_result.get("workflow_status"))

print("\nMarket Data Available:")
print(workflow_result.get("data_ok"))

print("\nNews Available:")
print(workflow_result.get("news_ok"))

print("\nSimulated Recommendation:")
print(workflow_result.get("recommendation"))

print("\nFINAL RESEARCH REPORT")
print(workflow_result.get("final_report"))



## 9. Results and Observations

The project successfully demonstrates the integration of stock market data retrieval, financial news retrieval, semantic search, and multi-agent workflow orchestration.

The following observations were made during testing:

1. Historical stock market information was successfully retrieved using Yahoo Finance.
2. Financial news articles were collected and converted into LangChain documents.
3. Sentence Transformer embeddings were generated locally.
4. FAISS successfully performed similarity-based news retrieval.
5. Three LangChain prompt templates were created and imported successfully.
6. LangGraph executed five specialized workflow nodes.
7. Conditional routing and shared state management were demonstrated.
8. The Streamlit application successfully displayed stock analysis and retrieved news.
9. The prototype was tested with RELIANCE.NS, TCS.NS, and INFY.NS.

The current implementation produces rule-based simulated recommendations. LLM-based reasoning remains pending API integration.



## 10. Limitations

The current implementation has the following limitations:

1. The application depends on Yahoo Finance data availability.
2. Financial news coverage may be incomplete or unavailable for some tickers.
3. The current recommendation mechanism uses basic historical percentage-change rules.
4. The LangChain prompt templates have not yet been connected to a live LLM.
5. The news analysis currently displays retrieved information rather than performing genuine LLM-based sentiment analysis.
6. The application does not include historical strategy backtesting.
7. The current prototype is intended for educational purposes and not actual investment decisions.

### Future Improvements

- Integrate a working LLM API.
- Implement genuine AI-based market and news analysis.
- Add structured output parsing.
- Improve financial sentiment analysis.
- Introduce historical backtesting and evaluation.
- Add additional market indicators and visualizations.



## 11. Conclusion

The AI Stock Market Assistant demonstrates how LangChain, LangGraph, Retrieval-Augmented Generation, FAISS, and Streamlit can be combined to develop a financial research application.

The project successfully implements stock market data retrieval, financial news processing, semantic search, prompt templates, shared state management, and conditional multi-agent workflow execution.

The developed Streamlit prototype provides an interactive interface for exploring stock information and viewing simulated recommendations.

The project establishes a foundation for further development through live LLM integration and more advanced financial reasoning capabilities.

**Disclaimer:** The application is an academic prototype. Its simulated recommendations should not be considered personalized financial advice.
